# DICOM-to-BIDS conversion using `heudiconv` and `singularity`

* This notebook creates jobs for each participant and executes them by looping across participants in this notebook or by creating slurm jobs that can be run in parallel on the slurm cluser
 

#### HISTORY

* 9/9/21 dcosme - initial code

## Do the DICOM to BIDS conversion for new data using existing config setup

* Now we have set up a mapping and edited the `heuristics.py` file we can do dicom2bids conversion with the following Singularity command.

* __N.B.__ Make sure the source folder in:
   ```
   /fmriDataRaw/fmri_data_raw/bbprime
   ```

* The only thing that needs to be changed is the subject identifier:
   ```
   -s BPP00
   ```
   
   it should match the folder name of the subject's data in
   ```
   /fmriDataRaw/fmri_data_raw/bbprime
   ```

## Loop through participants
The code below creates a job that can be executed in this notebook

### Define variables

In [ ]:
import os

project = 'geoscan_v2'
job_dir = os.path.join('/data00/projects/' + project + '/scripts/BIDS/jobs')
slurm_dir = job_dir

subs = ['GEO004']

job_file_template = r'''#!/bin/bash
singularity run --cleanenv \
    -B /data00/projects/geoscan_v2:/base  \
    -B /fmriDataRaw/fmri_data_raw:/raw \
    /data00/tools/singularity_images/heudiconv_0.8.0 \
    -d /raw/geoscan/T2/{subject}_T2/*.dcm \
    -o /base/data/bids_data/ \
    -f /base/scripts/BIDS/heudiconv/code/heuristic.py -ss t2 -s {ID} -c dcm2niix -b --overwrite
'''

NameError: name 'project_dir' is not defined

### Loop through specified participants

In [8]:
# make the job directory if it doens't exist
if os.path.exists(job_dir) == False:
    os.mkdir(job_dir)

# loop over participants
for s in subs:
    print('-------------- Creating: {} -------------'.format(s))
    
    file_path = os.path.join(job_dir, 'heudiconv_{}.job').format(s)
    print(file_path)

    with open(file_path.format(s), 'w') as job:
        job.write(job_file_template.format(ID=s, subject='{subject}'))
    
    !bash $file_path
    print('~~~~=====~~~~ Sub {} Completed ~~~~=====~~~~'.format(s))


-------------- Creating: GEO004 -------------
/data00/projects/geoscan_v2/scripts/BIDS/jobs/heudiconv_GEO004.job
INFO: Running heudiconv version 0.8.0 latest Unknown
INFO: Need to process 1 study sessions
INFO: PROCESSING STARTS: {'subject': 'GEO004', 'outdir': '/base/data/bids_data/', 'session': 't2'}
INFO: Processing 2419 dicoms
INFO: Reloading existing filegroup.json because /base/data/bids_data/.heudiconv/GEO004/ses-t2/info/GEO004_ses-t2.edit.txt exists
INFO: Doing conversion using dcm2niix
INFO: Converting /base/data/bids_data/sub-GEO004/ses-t2/anat/sub-GEO004_ses-t2_T1w (160 DICOMs) -> /base/data/bids_data/sub-GEO004/ses-t2/anat . Converter: dcm2niix . Output types: ('nii.gz',)
250522-16:46:12,733 nipype.utils WARNING:
	 Could not check for version updates: 
Connection to server could not be made
Connection to server could not be made
250522-16:46:12,739 nipype.workflow INFO:
	 [Node] Setting-up "convert" in "/tmp/dcm2niixvft5q1xj/convert".
INFO: [Node] Setting-up "convert" in "/

## Submit jobs using slurm
The code below creates a job that can be run using slurm

### Define variables

In [ ]:
project = 'Enter your project name here'
project_dir = os.path.join('/data00/projects/', project)
slurm_dir = os.path.join(project_dir, 'scripts/BIDS/jobs/heudiconv')
os.makedirs(slurm_dir, exist_ok=True)
subs = ['']

job_file_template = r'''#!/bin/bash
#SBATCH --job-name=heudiconv_{ID}.job
#SBATCH --output=out/heudiconv_{ID}.out
#SBATCH --error=out/heudiconv_{ID}.err
#SBATCH --time=02:00

srun singularity run --cleanenv \
    -B /data00/projects/{Your Project}:/base  \
    -B /fmriDataRaw/fmri_data_raw:/raw \
    /data00/tools/singularity_images/heudiconv_0.8.0 \
    -d /raw/{Your Project}/{ID}/*/*.dcm \
    -o /base/data/bids_data/ \
    -f heudiconv/code/heuristic.py -s {ID} -c dcm2niix -b --overwrite
'''

### Loop through specified participants

In [3]:
for s in subs:
    print('-------------- Creating: {} -------------'.format(s))
    
    file_path = os.path.join(slurm_dir, 'heudiconv_{}.job').format(s)
    print(file_path)

    with open(file_path.format(s), 'w') as job:
        job.write(job_file_template.format(ID=s, subject='{subject}'))


-------------- Creating: GEO004 -------------


NameError: name 'slurm_dir' is not defined

### Schedule the jobs on slurm

Login to the slurm cluster:

```
ssh <JANUS_UN>@asc.upenn.edu@cls000
```

This will give you a terminal on the SLURM master node where you can look at the process queue and schedule jobs by pasting the output from the next chunk

In [ ]:
print(f"cd {slurm_dir}")
for s in subs:
    print(f"sbatch -D {slurm_dir} -c 8 heudiconv_{s}.job")

In [ ]:
!ls /data00/slurm_jobs/slurm_bbprime/